#Assignment-5: Naive Bayes Classifier

Import required libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.datasets import load_iris
from sklearn.preprocessing import LabelEncoder # Add LabelEncoder for target variable
print('Libraries imported')


Libraries imported


##Load dataset
Financial Data for Fraud Detection and Risk Analysis

In [2]:
df = pd.read_csv('/content/FraudShield_Banking_Data (1).csv')
df.head()

,Transaction_ID,Customer_ID,Transaction_Amount (in Million),Transaction_Time,Transaction_Date,Transaction_Type,Merchant_ID,Merchant_Category,Transaction_Location,Customer_Home_Location,...,Daily_Transaction_Count,Weekly_Transaction_Count,Avg_Transaction_Amount (in Million),Max_Transaction_Last_24h (in Million),Is_International_Transaction,Is_New_Merchant,Failed_Transaction_Count,Unusual_Time_Transaction,Previous_Fraud_Count,Fraud_Label
0,431438.0,24239.0,6.0,10:54,2025-03-08,POS,97028.0,ATM,Singapore,Lahore,...,4.0,17.0,2.0,4.0,Yes,Yes,0.0,No,1.0,Normal
1,902451.0,77250.0,9.0,19:23,2025-01-17,ATM,27515.0,ATM,Singapore,Lahore,...,4.0,9.0,5.0,8.0,Yes,Yes,1.0,No,1.0,Normal
2,223410.0,34294.0,3.0,10:20,2025-04-30,POS,13810.0,Electronics,Faisalabad,Faisalabad,...,5.0,18.0,5.0,8.0,Yes,No,0.0,Yes,1.0,Normal
3,145626.0,92041.0,1.0,14:11,2025-02-21,Online,10501.0,Grocery,London,Karachi,...,6.0,18.0,5.0,1.0,No,Yes,2.0,Yes,1.0,Normal
4,414637.0,71578.0,1.0,04:12,2025-04-11,Online,53569.0,Electronics,Singapore,Islamabad,...,3.0,18.0,4.0,3.0,No,Yes,1.0,No,1.0,Normal


In [3]:

df.shape
df.info()
df.describe()
print('\nClass Distribution:')
print(df['Fraud_Label'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 25 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Transaction_ID                         49997 non-null  float64
 1   Customer_ID                            49990 non-null  float64
 2   Transaction_Amount (in Million)        49991 non-null  float64
 3   Transaction_Time                       49991 non-null  object 
 4   Transaction_Date                       49997 non-null  object 
 5   Transaction_Type                       49996 non-null  object 
 6   Merchant_ID                            49993 non-null  float64
 7   Merchant_Category                      49991 non-null  object 
 8   Transaction_Location                   49994 non-null  object 
 9   Customer_Home_Location                 49996 non-null  object 
 10  Distance_From_Home                     49998 non-null  float64
 11  De

In [4]:
df.dropna(inplace=True)

# Separate target variable before feature engineering
y = df['Fraud_Label']
X = df.drop('Fraud_Label', axis=1)

# Drop identifier columns that are not directly useful for GaussianNB
X = X.drop(['Transaction_ID', 'Customer_ID', 'Merchant_ID', 'Device_ID', 'IP_Address'], axis=1)

# Feature Engineering for Transaction_Time and Transaction_Date
X['Transaction_Time_Hour'] = pd.to_datetime(X['Transaction_Time'], format='%H:%M').dt.hour
X['Transaction_Date_Month'] = pd.to_datetime(X['Transaction_Date']).dt.month
X['Transaction_Date_DayOfWeek'] = pd.to_datetime(X['Transaction_Date']).dt.dayofweek

# Drop original time and date columns
X = X.drop(['Transaction_Time', 'Transaction_Date'], axis=1)

# Identify categorical columns for one-hot encoding
categorical_cols = X.select_dtypes(include='object').columns

# Apply one-hot encoding
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Encode the target variable
le = LabelEncoder()
y = le.fit_transform(y)

X.head()

,Transaction_Amount (in Million),Distance_From_Home,Account_Balance (in Million),Daily_Transaction_Count,Weekly_Transaction_Count,Avg_Transaction_Amount (in Million),Max_Transaction_Last_24h (in Million),Failed_Transaction_Count,Previous_Fraud_Count,Transaction_Time_Hour,...,Transaction_Location_Multan,Transaction_Location_Singapore,Customer_Home_Location_Islamabad,Customer_Home_Location_Karachi,Customer_Home_Location_Lahore,Customer_Home_Location_Multan,Card_Type_Debit,Is_International_Transaction_Yes,Is_New_Merchant_Yes,Unusual_Time_Transaction_Yes
0,6.0,466.0,30.0,4.0,17.0,2.0,4.0,0.0,1.0,10,...,False,True,False,False,True,False,False,True,True,False
1,9.0,215.0,4.0,4.0,9.0,5.0,8.0,1.0,1.0,19,...,False,True,False,False,True,False,False,True,True,False
2,3.0,216.0,38.0,5.0,18.0,5.0,8.0,0.0,1.0,10,...,False,False,False,False,False,False,True,True,False,True
3,1.0,408.0,22.0,6.0,18.0,5.0,1.0,2.0,1.0,14,...,False,False,False,True,False,False,True,False,True,True
4,1.0,209.0,10.0,3.0,18.0,4.0,3.0,1.0,1.0,4,...,False,True,True,False,False,False,True,False,True,False


In [5]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)
print('Training size:', X_train.shape)
print('Testing size:', X_test.shape)

Training size: (34895, 36)
Testing size: (14955, 36)


##Building the Naive Bayes model

In [6]:

nb = GaussianNB()
nb.fit(X_train, y_train)
print('Model trained')

Model trained


##Predict

In [7]:

y_pred = nb.predict(X_test)
list(y_pred)[:10]

[np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1),
 np.int64(1)]

##Compare predictions

In [8]:
results = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
results['Status'] = results.apply(lambda r: 'Correct' if r.Actual == r.Predicted else 'Wrong', axis=1)
results.head(20)

,Actual,Predicted,Status
0,1,1,Correct
1,1,1,Correct
2,1,1,Correct
3,1,1,Correct
4,1,1,Correct
5,1,1,Correct
6,1,1,Correct
7,1,1,Correct
8,1,1,Correct
9,1,1,Correct


##Compute Accuracy & Evaluation Metrics

In [9]:
print('Accuracy:', accuracy_score(y_test, y_pred))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred))

Accuracy: 0.9518555667001003

Confusion Matrix:
 [[    0   720]
 [    0 14235]]

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00       720
           1       0.95      1.00      0.98     14235

    accuracy                           0.95     14955
   macro avg       0.48      0.50      0.49     14955
weighted avg       0.91      0.95      0.93     14955



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



##Conclusion

This experiment demonstrates a complete workflow for building and evaluating a Naive Bayes classifier for fraud detection using the provided financial transaction data.The model achieved a high overall accuracy, primarily driven by its excellent ability to correctly identify the majority class ('Normal' transactions). However, the confusion matrix and classification report reveal a significant issue: the model completely failed to predict any instances of class 0 (Fraud). This means all actual fraud cases (720 instances) were incorrectly classified as 'Normal' (False Positives in the confusion matrix, where 'Predicted' is 1 and 'Actual' is 0). The UndefinedMetricWarning confirms this, as precision for class 0 is 0.0 because there were no true positive predictions for this class.